# Geographic Data Exploration

## Overview

This notebook examines the geographic metadata in the mechanical engineering job dataset
produced by Phase 1. Using `study_data_ME_ONLY.parquet` (~1.17M records, 2010–2022,
ME O*NET code 17-2141.00), we assess completeness and variation in eight geographic
fields to evaluate feasibility of spatial/regional analysis.

## Fields Analyzed

| Field | Type | Description |
|---|---|---|
| CanonCity | Categorical | Canonicalized city name |
| CanonCountry | Categorical | Canonicalized country |
| CanonState | Categorical | Canonicalized state/region |
| Latitude | Numeric | Job location latitude |
| Longitude | Numeric | Job location longitude |
| DivisionCode | Categorical | Division code of job location |
| CanonCounty | Categorical | Canonicalized county name |
| CanonPostalCode | Categorical | Canonicalized postal/ZIP code |

## Sections

1. **Missing Value Analysis** — overall rates, year-over-year, co-occurrence
2. **Descriptive Statistics** — unique values and top-N distributions
3. **Completeness Tiers** — classify records by geographic data quality

## Checkpointing

Intermediate computed statistics are saved to `checkpoints/` as CSVs. Re-running the
notebook after an interruption reloads from checkpoints rather than recomputing.
Checkpoints are dissolved (deleted) in the final cell once the full run is complete.

## Setup and Configuration

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import json
import shutil
import nbformat
from pathlib import Path
import sys

# Project root is one level up from data_exploration/
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

config_path = project_root / 'config' / 'analysis_config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

phase1_config = config['phase_1']
BASE_PATH = Path(phase1_config['base_path'])
OUTPUT_DIR = phase1_config['output_directory']
PARQUET_PATH = BASE_PATH / OUTPUT_DIR / phase1_config['output_file']
NOTEBOOK_DIR = Path.cwd()

print('✓ Imports successful')
print(f'Project root    : {project_root}')
print(f'Input parquet   : {PARQUET_PATH}')
print(f'Output directory: {NOTEBOOK_DIR}')

✓ Imports successful
Project root    : /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022
Input parquet   : /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/study_data_ME_ONLY.parquet
Output directory: /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022/data_exploration


## Checkpoint Utilities

In [2]:
CHECKPOINT_DIR = NOTEBOOK_DIR / 'checkpoints'


def save_checkpoint(df, name):
    """Save a Polars or pandas DataFrame as CSV to the checkpoint directory."""
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    path = CHECKPOINT_DIR / f'{name}.csv'
    if isinstance(df, pl.DataFrame):
        df.write_csv(path)
    else:
        df.to_csv(path, index=False)
    print(f'  ✓ Checkpoint saved: {path.name}')


def load_checkpoint(name):
    """Return Path if checkpoint CSV exists, else None."""
    path = CHECKPOINT_DIR / f'{name}.csv'
    return path if path.exists() else None


def dissolve_checkpoints():
    """Remove the checkpoints directory and all its contents."""
    if CHECKPOINT_DIR.exists():
        shutil.rmtree(CHECKPOINT_DIR)
        print('✓ Checkpoints dissolved')
    else:
        print('No checkpoints directory found — nothing to dissolve')


print('✓ Checkpoint utilities ready')
print(f'  Checkpoint directory: {CHECKPOINT_DIR}')

✓ Checkpoint utilities ready
  Checkpoint directory: /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022/data_exploration/checkpoints


## Load Geographic Data

We read only the 10 target columns from the parquet to minimize memory usage.
The parquet contains 54 columns total; loading a subset is handled natively by Polars.

In [3]:
GEO_FIELDS = [
    'CanonCity', 'CanonCountry', 'CanonState',
    'Latitude', 'Longitude', 'DivisionCode',
    'CanonCounty', 'CanonPostalCode'
]
LOAD_COLS = ['JobID', 'Year'] + GEO_FIELDS

print('=' * 80)
print('LOADING GEOGRAPHIC DATA')
print('=' * 80)
print(f'\nSource: {PARQUET_PATH}')
print(f'Columns requested: {LOAD_COLS}')

# Validate columns exist before loading (scan_parquet reads only metadata)
schema = pl.scan_parquet(PARQUET_PATH).schema
missing_cols = [c for c in LOAD_COLS if c not in schema]
if missing_cols:
    raise ValueError(f'Columns not found in parquet: {missing_cols}')
print(f'\n✓ All {len(LOAD_COLS)} columns confirmed in parquet schema')

data = pl.read_parquet(PARQUET_PATH, columns=LOAD_COLS)

print(f'\n✓ Loaded {len(data):,} records')
print('\nColumn dtypes:')
for col in data.columns:
    dtype = data[col].dtype
    print(f'  {col:<25} {dtype}')

LOADING GEOGRAPHIC DATA

Source: /Volumes/My Book/BurningGlassData/ASME_2026_PAPER/study_data_ME_ONLY.parquet
Columns requested: ['JobID', 'Year', 'CanonCity', 'CanonCountry', 'CanonState', 'Latitude', 'Longitude', 'DivisionCode', 'CanonCounty', 'CanonPostalCode']


/var/folders/z2/mk22tyls1_jfd89l0zs_f3qm0000gn/T/ipykernel_7354/1924204935.py:15: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = pl.scan_parquet(PARQUET_PATH).schema



✓ All 10 columns confirmed in parquet schema

✓ Loaded 1,166,271 records

Column dtypes:
  JobID                     String
  Year                      Int32
  CanonCity                 String
  CanonCountry              String
  CanonState                String
  Latitude                  String
  Longitude                 String
  DivisionCode              String
  CanonCounty               String
  CanonPostalCode           String


## Missing Value Analysis

We assess how frequently each geographic field is null, both overall and by year.
Understanding missingness patterns helps determine which fields are reliable enough
for spatial analysis and whether missingness varies across the study period.

In [4]:
print('=' * 80)
print('OVERALL MISSING VALUE ANALYSIS')
print('=' * 80)

ckpt = load_checkpoint('missing_value_stats')

if ckpt:
    print(f'\nLoading from checkpoint: {ckpt.name}')
    missing_df = pl.read_csv(ckpt)
else:
    total = len(data)
    rows = []
    for field in GEO_FIELDS:
        missing_count = int(data[field].is_null().sum())
        missing_pct = round((missing_count / total) * 100, 4)
        present_count = total - missing_count
        present_pct = round(100 - missing_pct, 4)
        rows.append({
            'field': field,
            'missing_count': missing_count,
            'missing_pct': missing_pct,
            'present_count': present_count,
            'present_pct': present_pct,
        })
    missing_df = pl.DataFrame(rows)
    save_checkpoint(missing_df, 'missing_value_stats')

total = len(data)
print(f'\nTotal records: {total:,}\n')
hdr = 'Field'.ljust(25) + 'Missing'.rjust(12) + 'Missing %'.rjust(12) + 'Present'.rjust(12) + 'Present %'.rjust(12)
print(hdr)
print('-' * 80)
for row in missing_df.iter_rows(named=True):
    field = row['field']
    mc = row['missing_count']
    mp = row['missing_pct']
    pc = row['present_count']
    pp = row['present_pct']
    print(f'{field:<25} {mc:>12,} {mp:>11.2f}% {pc:>12,} {pp:>11.2f}%')

OVERALL MISSING VALUE ANALYSIS
  ✓ Checkpoint saved: missing_value_stats.csv

Total records: 1,166,271

Field                         Missing   Missing %     Present   Present %
--------------------------------------------------------------------------------
CanonCity                       17,529        1.50%    1,148,742       98.50%
CanonCountry                         4        0.00%    1,166,267      100.00%
CanonState                         732        0.06%    1,165,539       99.94%
Latitude                        16,861        1.45%    1,149,410       98.55%
Longitude                       16,861        1.45%    1,149,410       98.55%
DivisionCode                   809,246       69.39%      357,025       30.61%
CanonCounty                     17,699        1.52%    1,148,572       98.48%
CanonPostalCode                 20,835        1.79%    1,145,436       98.21%


In [5]:
print('=' * 80)
print('MISSING VALUES BY YEAR')
print('=' * 80)

ckpt = load_checkpoint('missing_by_year')

if ckpt:
    print(f'\nLoading from checkpoint: {ckpt.name}')
    missing_by_year_df = pl.read_csv(ckpt)
else:
    agg_exprs = [pl.len().alias('total')] + [
        (pl.col(f).is_null().sum() / pl.len() * 100).round(2).alias(f'{f}_missing_pct')
        for f in GEO_FIELDS
    ]
    missing_by_year_df = data.group_by('Year').agg(agg_exprs).sort('Year')
    save_checkpoint(missing_by_year_df, 'missing_by_year')

SHORT_NAMES = ['City', 'Country', 'State', 'Lat', 'Lon', 'Div', 'County', 'Postal']

print('\nMissing rate (%) by year:\n')
hdr = 'Year'.rjust(6) + 'Jobs'.rjust(10) + ' ' + ' '.join(s.rjust(9) for s in SHORT_NAMES)
print(hdr)
print('-' * len(hdr))

for row in missing_by_year_df.iter_rows(named=True):
    year = int(row['Year'])
    total = int(row['total'])
    line = f'{year:>6} {total:>9,}'
    for f in GEO_FIELDS:
        val = row[f'{f}_missing_pct']
        line += f' {val:>9.1f}'
    print(line)

MISSING VALUES BY YEAR
  ✓ Checkpoint saved: missing_by_year.csv

Missing rate (%) by year:

  Year      Jobs      City   Country     State       Lat       Lon       Div    County    Postal
------------------------------------------------------------------------------------------------
  2010    56,102       1.7       0.0       0.0       1.7       1.7      69.9       1.7       1.7
  2011    82,042       1.4       0.0       0.0       1.4       1.4      71.7       1.5       1.4
  2012    82,478       1.4       0.0       0.0       1.4       1.4      71.4       1.4       1.4
  2013    87,159       1.8       0.0       0.0       1.7       1.7      72.8       1.8       1.8
  2014    84,173       1.6       0.0       0.1       1.5       1.5      71.5       1.6       1.7
  2015    95,569       0.7       0.0       0.1       0.7       0.7      67.8       0.7       0.8
  2016    83,584       0.6       0.0       0.0       0.6       0.6      66.8       0.6       0.7
  2017    86,887       0.6       0

In [6]:
print('=' * 80)
print('MISSINGNESS CO-OCCURRENCE MATRIX')
print('=' * 80)
print('Diagonal = field own null count; off-diagonal = count where BOTH fields are null\n')

n = len(GEO_FIELDS)
matrix = np.zeros((n, n), dtype=int)

for i, f1 in enumerate(GEO_FIELDS):
    for j, f2 in enumerate(GEO_FIELDS):
        if i == j:
            matrix[i, j] = int(data[f1].is_null().sum())
        else:
            matrix[i, j] = data.filter(
                pl.col(f1).is_null() & pl.col(f2).is_null()
            ).height

labels = [f.replace('Canon', '').replace('DivisionCode', 'DivCode') for f in GEO_FIELDS]
co_df = pd.DataFrame(matrix, index=labels, columns=labels)
print(co_df.to_string())
print()
print('Note: high off-diagonal values indicate these fields are jointly missing,')
print('suggesting a shared source column rather than independent imputation.')

MISSINGNESS CO-OCCURRENCE MATRIX
Diagonal = field own null count; off-diagonal = count where BOTH fields are null

             City  Country  State  Latitude  Longitude  DivCode  County  PostalCode
City        17529        0    732     16813      16813    17529   17529       17469
Country         0        4      0         0          0        4       0           0
State         732        0    732        16         16      732     732         672
Latitude    16813        0     16     16861      16861    16861   16861       16861
Longitude   16813        0     16     16861      16861    16861   16861       16861
DivCode     17529        4    732     16861      16861   809246   17699       19900
County      17529        0    732     16861      16861    17699   17699       17528
PostalCode  17469        0    672     16861      16861    19900   17528       20835

Note: high off-diagonal values indicate these fields are jointly missing,
suggesting a shared source column rather than independ

## Descriptive Statistics

For categorical fields we report unique value counts and the top-20 most common
values. For numeric fields (Latitude, Longitude) we report distribution statistics
and check for suspicious values (zero coordinates, out-of-range values).

In [7]:
print('=' * 80)
print('CATEGORICAL FIELD DESCRIPTIVE STATISTICS')
print('=' * 80)

CATEGORICAL_FIELDS = [
    'CanonCity', 'CanonCountry', 'CanonState',
    'DivisionCode', 'CanonCounty', 'CanonPostalCode'
]

ckpt = load_checkpoint('categorical_stats')

if ckpt:
    print(f'\nLoading summary from checkpoint: {ckpt.name}')
    cat_summary_df = pl.read_csv(ckpt)
    print(cat_summary_df)
else:
    summary_rows = []

    for field in CATEGORICAL_FIELDS:
        non_null_data = data.filter(pl.col(field).is_not_null())
        n_present = len(non_null_data)
        n_unique = non_null_data[field].n_unique()

        top20 = (
            non_null_data.group_by(field)
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(20)
            .with_columns(
                (pl.col('count') / n_present * 100).round(3).alias('pct')
            )
        )

        top_val = str(top20[field][0]) if len(top20) > 0 else ''
        top_cnt = int(top20['count'][0]) if len(top20) > 0 else 0
        top_pct = float(top20['pct'][0]) if len(top20) > 0 else 0.0

        summary_rows.append({
            'field': field,
            'n_unique': n_unique,
            'n_present': n_present,
            'top_value': top_val,
            'top_count': top_cnt,
            'top_pct': top_pct,
        })

        print()
        print('─' * 60)
        print(f'  {field}')
        print('─' * 60)
        print(f'  Unique values: {n_unique:,}  |  Present records: {n_present:,}')
        print()
        print('  ' + 'Value'.ljust(36) + 'Count'.rjust(10) + 'Share %'.rjust(10))
        print('  ' + '-' * 36 + '-' * 10 + '-' * 10)
        for top_row in top20.iter_rows(named=True):
            val_str = str(top_row[field])[:35]
            cnt = top_row['count']
            pct = top_row['pct']
            print(f'  {val_str:<36} {cnt:>10,} {pct:>10.3f}%')

    cat_summary_df = pl.DataFrame(summary_rows)
    save_checkpoint(cat_summary_df, 'categorical_stats')
    print()
    print()
    print('✓ Categorical summary:')
    print(cat_summary_df)

CATEGORICAL FIELD DESCRIPTIVE STATISTICS

────────────────────────────────────────────────────────────
  CanonCity
────────────────────────────────────────────────────────────
  Unique values: 8,835  |  Present records: 1,148,742

  Value                                    Count   Share %
  --------------------------------------------------------
  Houston                                  32,324      2.814%
  San Diego                                14,840      1.292%
  New York                                 13,744      1.196%
  Los Angeles                              12,918      1.125%
  Auburn Hills                             11,403      0.993%
  San Jose                                 10,802      0.940%
  Minneapolis                              10,613      0.924%
  Atlanta                                  10,445      0.909%
  Chicago                                   9,451      0.823%
  Phoenix                                   9,307      0.810%
  Irvine                       

In [8]:
print('=' * 80)
print('NUMERIC FIELD STATISTICS  (LATITUDE / LONGITUDE)')
print('=' * 80)

ckpt = load_checkpoint('numeric_stats')

if ckpt:
    print(f'\nLoading from checkpoint: {ckpt.name}')
    num_summary_df = pl.read_csv(ckpt)
    print(num_summary_df)
else:
    NUMERIC_FIELDS = ['Latitude', 'Longitude']
    num_rows = []

    for field in NUMERIC_FIELDS:
        # Cast to Float64 — these columns may be stored as strings in the parquet
        col = data[field].cast(pl.Float64, strict=False).drop_nulls()
        n_present = len(col)

        val_min = float(col.min())
        p5 = float(col.quantile(0.05))
        p25 = float(col.quantile(0.25))
        val_mean = float(col.mean())
        val_median = float(col.median())
        p75 = float(col.quantile(0.75))
        p95 = float(col.quantile(0.95))
        val_max = float(col.max())
        val_std = float(col.std())

        num_rows.append({
            'field': field,
            'n_present': n_present,
            'val_min': val_min,
            'p5': p5,
            'p25': p25,
            'val_mean': val_mean,
            'val_median': val_median,
            'p75': p75,
            'p95': p95,
            'val_max': val_max,
            'val_std': val_std,
        })

        print()
        print(f'  {field}')
        print('  ' + '─' * 50)
        print(f'  Present records : {n_present:>12,}')
        print(f'  Min             : {val_min:>12.4f}')
        print(f'  5th percentile  : {p5:>12.4f}')
        print(f'  25th percentile : {p25:>12.4f}')
        print(f'  Mean            : {val_mean:>12.4f}')
        print(f'  Median          : {val_median:>12.4f}')
        print(f'  75th percentile : {p75:>12.4f}')
        print(f'  95th percentile : {p95:>12.4f}')
        print(f'  Max             : {val_max:>12.4f}')
        print(f'  Std dev         : {val_std:>12.4f}')

    print()
    print('─' * 60)
    print('  Suspicious value checks:')
    print('─' * 60)

    # Cast to Float64 for comparisons as well
    lat = data['Latitude'].cast(pl.Float64, strict=False)
    lon = data['Longitude'].cast(pl.Float64, strict=False)
    both_present = data.with_columns([lat.alias('_lat'), lon.alias('_lon')]).filter(
        pl.col('_lat').is_not_null() & pl.col('_lon').is_not_null()
    )
    zero_zero = both_present.filter(
        (pl.col('_lat') == 0.0) & (pl.col('_lon') == 0.0)
    ).height
    invalid_lat = both_present.filter(pl.col('_lat').abs() > 90.0).height
    invalid_lon = both_present.filter(pl.col('_lon').abs() > 180.0).height

    print(f'  Records with Lat=0, Lon=0   : {zero_zero:>10,}')
    print(f'  Records with |Lat| > 90     : {invalid_lat:>10,}')
    print(f'  Records with |Lon| > 180    : {invalid_lon:>10,}')

    num_summary_df = pl.DataFrame(num_rows)
    save_checkpoint(num_summary_df, 'numeric_stats')
    print()
    print('✓ Numeric summary saved')

NUMERIC FIELD STATISTICS  (LATITUDE / LONGITUDE)

  Latitude
  ──────────────────────────────────────────────────
  Present records :    1,149,410
  Min             :       7.3270
  5th percentile  :      29.7194
  25th percentile :      34.0023
  Mean            :      38.1697
  Median          :      39.1072
  75th percentile :      42.1052
  95th percentile :      45.3078
  Max             :      71.3003
  Std dev         :       5.1659

  Longitude
  ──────────────────────────────────────────────────
  Present records :    1,149,410
  Min             :    -166.5380
  5th percentile  :    -122.2000
  25th percentile :    -105.2190
  Mean            :     -93.3592
  Median          :     -87.9445
  75th percentile :     -80.8571
  95th percentile :     -72.6454
  Max             :     166.6420
  Std dev         :      16.8934

────────────────────────────────────────────────────────────
  Suspicious value checks:
────────────────────────────────────────────────────────────
  Records 

## Geographic Completeness Tiers

We classify each record into one of five mutually exclusive tiers based on which
geographic fields are populated:

| Tier | Criteria |
|---|---|
| **Full** | All 8 geographic fields present |
| **Has-coordinates** | Both Latitude and Longitude present (text may be partial) |
| **Text-only** | At least one text field present; coordinates missing |
| **None** | All 8 geographic fields null |
| **Other** | Edge cases (e.g., only one of lat/lon present, no text) |

In [9]:
print('=' * 80)
print('GEOGRAPHIC COMPLETENESS TIERS')
print('=' * 80)

ckpt = load_checkpoint('completeness_tiers')

if ckpt:
    print(f'\nLoading from checkpoint: {ckpt.name}')
    tier_df = pl.read_csv(ckpt)
    print(tier_df)
else:
    TEXT_FIELDS = [
        'CanonCity', 'CanonCountry', 'CanonState',
        'DivisionCode', 'CanonCounty', 'CanonPostalCode'
    ]

    all_8_present = pl.all_horizontal(*[pl.col(f).is_not_null() for f in GEO_FIELDS])
    has_coords = pl.col('Latitude').is_not_null() & pl.col('Longitude').is_not_null()
    has_any_text = pl.any_horizontal(*[pl.col(f).is_not_null() for f in TEXT_FIELDS])
    all_geo_null = pl.all_horizontal(*[pl.col(f).is_null() for f in GEO_FIELDS])

    data_tiered = data.with_columns(
        pl.when(all_8_present).then(pl.lit('Full'))
        .when(has_coords).then(pl.lit('Has-coordinates'))
        .when(has_any_text).then(pl.lit('Text-only'))
        .when(all_geo_null).then(pl.lit('None'))
        .otherwise(pl.lit('Other'))
        .alias('completeness_tier')
    )

    tier_df = (
        data_tiered.group_by('completeness_tier')
        .agg(pl.len().alias('count'))
        .sort('count', descending=True)
        .with_columns(
            (pl.col('count') / len(data) * 100).round(2).alias('pct')
        )
    )

    print(f'\nOverall completeness (n={len(data):,}):\n')
    print('  ' + 'Tier'.ljust(20) + 'Count'.rjust(12) + 'Pct %'.rjust(10))
    print('  ' + '-' * 42)
    for row in tier_df.iter_rows(named=True):
        tier = row['completeness_tier']
        cnt = row['count']
        pct = row['pct']
        print(f'  {tier:<20} {cnt:>12,} {pct:>10.2f}%')

    # Verify tiers sum to total
    tier_total = tier_df['count'].sum()
    print(f'\n  Total: {tier_total:,}  (expected {len(data):,})')

    # Year breakdown for Full tier
    year_totals = data.group_by('Year').agg(pl.len().alias('total')).sort('Year')
    full_by_year = (
        data_tiered.filter(pl.col('completeness_tier') == 'Full')
        .group_by('Year').agg(pl.len().alias('count'))
        .sort('Year')
    )
    full_pct_df = full_by_year.join(year_totals, on='Year').with_columns(
        (pl.col('count') / pl.col('total') * 100).round(2).alias('full_pct')
    ).sort('Year')

    print()
    print()
    print('  Full-tier records by year:\n')
    print('  ' + 'Year'.rjust(6) + 'Total Jobs'.rjust(12) + 'Full'.rjust(12) + 'Full %'.rjust(10))
    print('  ' + '-' * 40)
    for row in full_pct_df.iter_rows(named=True):
        year = int(row['Year'])
        total = int(row['total'])
        full_cnt = int(row['count'])
        full_pct = row['full_pct']
        print(f'  {year:>8} {total:>12,} {full_cnt:>12,} {full_pct:>10.2f}%')

    save_checkpoint(tier_df, 'completeness_tiers')

GEOGRAPHIC COMPLETENESS TIERS

Overall completeness (n=1,166,271):

  Tier                       Count     Pct %
  ------------------------------------------
  Has-coordinates           793,320      68.02%
  Full                      356,090      30.53%
  Text-only                  16,861       1.45%

  Total: 1,166,271  (expected 1,166,271)


  Full-tier records by year:

    Year  Total Jobs        Full    Full %
  ----------------------------------------
      2010       56,102       16,902      30.13%
      2011       82,042       23,195      28.27%
      2012       82,478       23,555      28.56%
      2013       87,159       23,671      27.16%
      2014       84,173       23,972      28.48%
      2015       95,569       30,750      32.18%
      2016       83,584       27,691      33.13%
      2017       86,887       28,043      32.28%
      2018      103,852       31,928      30.74%
      2019      102,086       33,032      32.36%
      2020       68,675       22,839      33.26%

## Export Permanent Statistics

Two CSV files are written to `data_exploration/`:

- **`geo_missing_value_stats.csv`** — overall missing rates + per-year missing rates (wide format)
- **`geo_descriptive_stats.csv`** — categorical and numeric summaries in a unified schema

Checkpoints are dissolved in the next cell.

In [10]:
print('=' * 80)
print('EXPORTING PERMANENT STATISTICS')
print('=' * 80)

overall_path = NOTEBOOK_DIR / 'geo_missing_value_stats.csv'
desc_path = NOTEBOOK_DIR / 'geo_descriptive_stats.csv'

# ---------- geo_missing_value_stats.csv ----------
# Combine overall rates + per-year rates into a wide table (one row per field)

missing_overall = pl.read_csv(CHECKPOINT_DIR / 'missing_value_stats.csv')
missing_by_year = pl.read_csv(CHECKPOINT_DIR / 'missing_by_year.csv')

rows = []
for field in GEO_FIELDS:
    row = missing_overall.filter(pl.col('field') == field).to_dicts()[0]
    col_name = f'{field}_missing_pct'
    for yr_row in missing_by_year.iter_rows(named=True):
        year = int(yr_row['Year'])
        row[f'yr_{year}_missing_pct'] = yr_row[col_name]
    rows.append(row)

missing_export = pl.DataFrame(rows)
missing_export.write_csv(overall_path)
print(f'\n✓ Saved: {overall_path}')
print(f'  Rows: {len(missing_export)}, Columns: {len(missing_export.columns)}')

# ---------- geo_descriptive_stats.csv ----------
# Unified schema: categorical fields get null numeric columns; numeric fields get null categorical columns

cat_summary = pl.read_csv(CHECKPOINT_DIR / 'categorical_stats.csv')
num_summary = pl.read_csv(CHECKPOINT_DIR / 'numeric_stats.csv')

cat_aligned = cat_summary.with_columns([
    pl.lit('categorical').alias('field_type'),
    pl.lit(None).alias('val_min'),
    pl.lit(None).alias('val_max'),
    pl.lit(None).alias('val_mean'),
    pl.lit(None).alias('val_median'),
    pl.lit(None).alias('val_std'),
]).select([
    'field', 'field_type', 'n_present', 'n_unique',
    'top_value', 'top_count', 'top_pct',
    'val_min', 'val_max', 'val_mean', 'val_median', 'val_std'
])

num_aligned = (
    num_summary
    .select(['field', 'n_present', 'val_min', 'val_max', 'val_mean', 'val_median', 'val_std'])
    .with_columns([
        pl.lit('numeric').alias('field_type'),
        pl.lit(None).alias('n_unique'),
        pl.lit(None).alias('top_value'),
        pl.lit(None).alias('top_count'),
        pl.lit(None).alias('top_pct'),
    ])
    .select([
        'field', 'field_type', 'n_present', 'n_unique',
        'top_value', 'top_count', 'top_pct',
        'val_min', 'val_max', 'val_mean', 'val_median', 'val_std'
    ])
)

# Use pandas concat to handle mixed null types gracefully
desc_export = pd.concat(
    [cat_aligned.to_pandas(), num_aligned.to_pandas()],
    ignore_index=True
)
desc_export.to_csv(desc_path, index=False)

print(f'\n✓ Saved: {desc_path}')
print(f'  Rows: {len(desc_export)}, Columns: {len(desc_export.columns)}')
print()
print(desc_export.to_string(index=False))

EXPORTING PERMANENT STATISTICS

✓ Saved: /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022/data_exploration/geo_missing_value_stats.csv
  Rows: 8, Columns: 18

✓ Saved: /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022/data_exploration/geo_descriptive_stats.csv
  Rows: 8, Columns: 12

          field  field_type  n_present n_unique   top_value top_count top_pct  val_min  val_max   val_mean val_median    val_std
      CanonCity categorical    1148742     8835     Houston     32324   2.814     None     None       None       None       None
   CanonCountry categorical    1166267        2         USA   1166243  99.998     None     None       None       None       None
     CanonState categorical    1165539       56          CA    173338  14.872     None     None       None       None       None
   DivisionCode categorical     357025       41       47664     39593   11.09     None     None       None       None       None
    CanonCoun

In [11]:
# Dissolve checkpoints — this cell marks the notebook as fully run.
# Re-running from the top will rebuild checkpoints; re-running from here dissolves them.
dissolve_checkpoints()

print()
print('✓ Geographic exploration complete')
print()
print('Permanent outputs:')
print(f'  • {NOTEBOOK_DIR / "geo_missing_value_stats.csv"}')
print(f'  • {NOTEBOOK_DIR / "geo_descriptive_stats.csv"}')

✓ Checkpoints dissolved

✓ Geographic exploration complete

Permanent outputs:
  • /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022/data_exploration/geo_missing_value_stats.csv
  • /Users/mitchellgerhardt/Documents/Research/mech-eng-job-analysis-2010-to-2022/data_exploration/geo_descriptive_stats.csv


## Geographic Visualization

Two interactive maps using the ~98.5% of records that have coordinates.
Both filter to continental USA (lat 24–50, lon −125 to −66).
Hover for city/state details, zoom, and pan.

- **State choropleth** — total job postings per state (2010–2022)
- **Top 50 cities bubble map** — bubble size and color proportional to job count

In [12]:
import plotly.express as px
import numpy as np

# Full-name → 2-letter abbreviation fallback (BGT CanonState is usually already abbreviated)
STATE_ABBR = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC',
}

state_counts = (
    data.filter(pl.col('CanonState').is_not_null())
    .group_by('CanonState')
    .agg(pl.len().alias('job_count'))
    .sort('job_count', descending=True)
).to_pandas()

# Normalize to 2-letter codes if CanonState contains full names
sample_val = state_counts['CanonState'].iloc[0]
if len(sample_val) > 2:
    state_counts['state_code'] = state_counts['CanonState'].map(STATE_ABBR)
    state_counts = state_counts.dropna(subset=['state_code'])
else:
    state_counts['state_code'] = state_counts['CanonState']

# Keep only the 50 states + DC for the choropleth
us_states = state_counts[state_counts['state_code'].str.len() == 2].copy()

print(f'States in map: {len(us_states)}')
print(us_states.head(10).to_string(index=False))

# Log-scale color so CA (173k) doesn't drown out Wyoming (~1k).
# Viridis (purple→blue→teal→green→yellow) is used instead of YlOrRd because
# even the lowest states have thousands of jobs — all values land in the upper
# portion of a warm scale, making everything look the same shade of red.
us_states['log_jobs'] = np.log10(us_states['job_count'])
_lo, _hi = us_states['log_jobs'].min(), us_states['log_jobs'].max()
_tick_log  = [v for v in [2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5] if _lo <= v <= _hi]
_tick_text = [f'{int(round(10 ** v)):,}' for v in _tick_log]

fig = px.choropleth(
    us_states,
    locations='state_code',
    locationmode='USA-states',
    color='log_jobs',
    scope='usa',
    color_continuous_scale='Viridis',
    range_color=(_lo, _hi),
    title='Mechanical Engineering Job Postings by State (2010–2022)',
    labels={'state_code': 'State', 'CanonState': 'State'},
    hover_data={'CanonState': True, 'job_count': True, 'state_code': False, 'log_jobs': False},
)
fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(
        title='Job Postings',
        tickvals=_tick_log,
        ticktext=_tick_text,
    ),
)
fig.show()


States in map: 56
CanonState  job_count state_code
        CA     173338         CA
        TX      93222         TX
        MI      84753         MI
        OH      51360         OH
        NY      47509         NY
        IL      46708         IL
        MA      40464         MA
        PA      40362         PA
        FL      37006         FL
        WI      32122         WI


In [13]:
city_stats = (
    data.filter(
        pl.col('CanonCity').is_not_null() &
        pl.col('Latitude').is_not_null() &
        pl.col('Longitude').is_not_null()
    )
    .with_columns([
        pl.col('Latitude').cast(pl.Float64, strict=False),
        pl.col('Longitude').cast(pl.Float64, strict=False),
    ])
    .group_by(['CanonCity', 'CanonState'])
    .agg([
        pl.len().alias('job_count'),
        pl.col('Latitude').median().alias('lat'),
        pl.col('Longitude').median().alias('lon'),
    ])
    .sort('job_count', descending=True)
)

# Filter to continental USA and take top 50 cities
top_cities = city_stats.filter(
    (pl.col('lat') >= 24) & (pl.col('lat') <= 50) &
    (pl.col('lon') >= -125) & (pl.col('lon') <= -66)
).head(50).to_pandas()

print(f'Top 50 cities (continental USA):')
print(top_cities[['CanonCity', 'CanonState', 'job_count']].to_string(index=False))

# Log-scale color so mid-tier cities (Boston, Seattle) register clearly
# alongside the Houston/San Diego outliers at the top.
top_cities = top_cities.copy()
top_cities['log_jobs'] = np.log10(top_cities['job_count'])
_lo_c, _hi_c = top_cities['log_jobs'].min(), top_cities['log_jobs'].max()
_tick_log_c  = [v for v in [3.0, 3.5, 4.0, 4.5, 5.0] if _lo_c <= v <= _hi_c]
_tick_text_c = [f'{int(round(10 ** v)):,}' for v in _tick_log_c]

fig = px.scatter_geo(
    top_cities,
    lat='lat',
    lon='lon',
    size='job_count',
    hover_name='CanonCity',
    hover_data={'CanonState': True, 'job_count': True, 'lat': False, 'lon': False, 'log_jobs': False},
    scope='usa',
    size_max=40,
    color='log_jobs',
    color_continuous_scale='YlOrRd',
    range_color=(_lo_c, _hi_c),
    title='Top 50 Cities by Mechanical Engineering Job Postings (2010–2022)',
    labels={'log_jobs': 'Job Postings', 'CanonState': 'State'},
)
fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(
        title='Job Postings',
        tickvals=_tick_log_c,
        ticktext=_tick_text_c,
    ),
)
fig.show()


Top 50 cities (continental USA):
     CanonCity CanonState  job_count
       Houston         TX      32251
     San Diego         CA      14839
      New York         NY      13744
   Los Angeles         CA      12918
  Auburn Hills         MI      11403
      San Jose         CA      10799
   Minneapolis         MN      10612
       Atlanta         GA      10410
       Chicago         IL       9451
       Phoenix         AZ       9294
        Irvine         CA       8317
        Dallas         TX       8049
        Austin         TX       7821
       Seattle         WA       7634
    Huntsville         AL       7588
   Santa Clara         CA       7525
    Cincinnati         OH       7499
       Detroit         MI       7477
        Denver         CO       7477
 San Francisco         CA       7374
       Orlando         FL       7174
   Albuquerque         NM       6868
        Boston         MA       6611
      Portland         OR       5979
    Washington         DC       5952
     